In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json
from PIL import Image
import os

# preprocessing data

In [4]:
#create validation caption
with open("/kaggle/input/image-processing-thai-language-image-captioning/capgen_v1.0_val.json", "r", encoding="utf-8") as file:
    data_val = json.load(file)

# Convert JSON to a list of tuples (image_path, index, caption)
records = []
for image_path, captions in data_val.items():
    for idx, caption in enumerate(captions):
        records.append((image_path, idx, caption))

# Create DataFrame
caption_val = pd.DataFrame(records, columns=["image_path", "caption_index", "caption_text"])

In [5]:
#create training caption
with open("/kaggle/input/image-processing-thai-language-image-captioning/capgen_v1.0_train.json", "r", encoding="utf-8") as file:
    data_train = json.load(file)

# Convert JSON to a list of tuples (image_path, index, caption)
records = []
for image_path, captions in data_train.items():
    for idx, caption in enumerate(captions):
        records.append((image_path, idx, caption))

# Create DataFrame
caption_train = pd.DataFrame(records, columns=["image_path", "caption_index", "caption_text"])

In [6]:
#img = Image.open(image_path)
caption_train['from'] = caption_train["image_path"].apply(lambda x: x.split('/')[0])
caption_train['img_id'] = caption_train["image_path"].apply(lambda x: x.split('/')[-1])

In [7]:
caption_train

,image_path,caption_index,caption_text,from,img_id
0,coco/train2017/000000373716.jpg,0,ผู้หญิงสวมเสื้อแขนยาวสีขาวและเด็กนั่งเล่นกับสุ...,coco,000000373716.jpg
1,coco/train2017/000000373716.jpg,1,สาวคนนึงกำลังพาเด็กมานั่งเล่นอยู่ภายในสนามหญ้า...,coco,000000373716.jpg
2,coco/train2017/000000373716.jpg,2,ภาพขาวดำ ผู้หญิงนั่งบนพื้นอุ้มเด็กบนตัก ข้าง ๆ...,coco,000000373716.jpg
3,coco/train2017/000000196888.jpg,0,สีน้ำตาลตัวเล็กกำลังกินอาหารอยู่บนจานกระดาษสีข...,coco,000000196888.jpg
4,coco/train2017/000000196888.jpg,1,นกน้อยตัวหนึ่งกำลังจิกกินเศษอาหารที่วางทิ้งไว้...,coco,000000196888.jpg
...,...,...,...,...,...
430280,ipu24/train/food/28002.jpg,1,ขนมสับปะรดที่ใส่หมูสับใส่ในจานสีขาวและมีผักสวน...,ipu24,28002.jpg
430281,ipu24/train/food/28002.jpg,2,ในจานสีขาวมีสับปะรดหั่นเป็นชิ้นสามเหลี่ยมเล็ก ...,ipu24,28002.jpg
430282,ipu24/train/food/28003.jpg,0,ปลาดุกย่างเสียบไม้หลายตัวตั้งอยู่ในถาดมีผักแช่...,ipu24,28003.jpg
430283,ipu24/train/food/28003.jpg,1,ปลาดุกปิ้งจำนวนมากที่วางอยู่ในถาดอะลูมิเนียมทร...,ipu24,28003.jpg


In [8]:
caption_train_sub = caption_train[caption_train['caption_index']==0]
caption_train_sub = caption_train[caption_train['from']=='ipu24']

In [9]:
caption_train_sub = caption_train_sub.drop('caption_index', axis=1)
caption

'ปลาดุกย่างหลายตัวเสียบไม้วางอยู่บนตะแกรงที่มีถาดรองและด้านหน้ามีกะละมังที่แช่สะเดา'

In [10]:
caption_train_sub['from'].value_counts()

from
ipu24    85241
Name: count, dtype: int64

In [11]:
import os
import pandas as pd
from tqdm import tqdm  # Import tqdm for progress bar

# Function to load a single image based on row info
def load_image(row):
    # Define the base path for each source type
    if row['from'] == 'ipu24':
        if row['image_path'].split('/')[2] == 'travel':
            img_path = os.path.join('/kaggle/input/image-processing-thai-language-image-captioning/train/train/travel', row['img_id'])
        elif row['image_path'].split('/')[2] == 'food':
            img_path = os.path.join('/kaggle/input/image-processing-thai-language-image-captioning/train/train/food', row['img_id'])
    elif row['from'] == 'coco':
        img_path = os.path.join('/kaggle/input/coco-2017-dataset/coco2017/train2017', row['img_id'])

    return img_path


# Function to load images in batches and merge them into the same dataframe with a progress bar for the entire dataset
def load_images_with_progress(df, batch_size=100):
    all_batches = []  # List to store all processed batches
    
    # Create a progress bar for the entire dataset
    with tqdm(total=len(df), desc="Loading Images", unit="image") as pbar:
        for start in range(0, len(df), batch_size):
            batch = df.iloc[start:start + batch_size].copy()  # Create a copy of the batch to avoid SettingWithCopyWarning
            batch.loc[:, "new_image_path"] = batch.apply(load_image, axis=1)  # Load images for the batch
            all_batches.append(batch)  # Append the batch to the list
            
            # Update the progress bar after each batch
            pbar.update(batch_size)
    
    # Concatenate all batches back into a single dataframe
    df = pd.concat(all_batches, ignore_index=True)
    return df

In [12]:
train_real = load_images_with_progress(caption_train_sub, batch_size = 100)

Loading Images: 85300image [00:01, 51315.65image/s]                        


In [13]:
train_real.to_csv("train_real_with_path.csv")

In [18]:
train_real.head()

,image_path,caption_text,from,img_id,new_image_path
0,ipu24/train/travel/00000.jpg,รูปปั้นพญานาคตัวสีทองมีดอกบัวอยู่ด้านข้างอยู่ใ...,ipu24,00000.jpg,/kaggle/input/image-processing-thai-language-i...
1,ipu24/train/travel/00000.jpg,รูปปั้นมังกรที่อยู่ภายในศาลเจ้าจีนและมีผ้าติดต...,ipu24,00000.jpg,/kaggle/input/image-processing-thai-language-i...
2,ipu24/train/travel/00000.jpg,รูปปั้นมังกร มีหนวดสีทอง เขี้ยวและฟันสีขาว ลำต...,ipu24,00000.jpg,/kaggle/input/image-processing-thai-language-i...
3,ipu24/train/travel/00001.jpg,โบราณสถานที่มีเจดีย์สีน้ำตาลดำมีต้นไม้หลายต้นด...,ipu24,00001.jpg,/kaggle/input/image-processing-thai-language-i...
4,ipu24/train/travel/00001.jpg,สร้างโบราณสถานขนาดใหญ่ที่อยู่ตรงพื้นที่เต็มไปด...,ipu24,00001.jpg,/kaggle/input/image-processing-thai-language-i...


In [19]:
train_real_fr = train_real.drop(['image_path', 'from', 'img_id',], axis=1)
train_real_fr

,caption_text,new_image_path
0,รูปปั้นพญานาคตัวสีทองมีดอกบัวอยู่ด้านข้างอยู่ใ...,/kaggle/input/image-processing-thai-language-i...
1,รูปปั้นมังกรที่อยู่ภายในศาลเจ้าจีนและมีผ้าติดต...,/kaggle/input/image-processing-thai-language-i...
2,รูปปั้นมังกร มีหนวดสีทอง เขี้ยวและฟันสีขาว ลำต...,/kaggle/input/image-processing-thai-language-i...
3,โบราณสถานที่มีเจดีย์สีน้ำตาลดำมีต้นไม้หลายต้นด...,/kaggle/input/image-processing-thai-language-i...
4,สร้างโบราณสถานขนาดใหญ่ที่อยู่ตรงพื้นที่เต็มไปด...,/kaggle/input/image-processing-thai-language-i...
...,...,...
85236,ขนมสับปะรดที่ใส่หมูสับใส่ในจานสีขาวและมีผักสวน...,/kaggle/input/image-processing-thai-language-i...
85237,ในจานสีขาวมีสับปะรดหั่นเป็นชิ้นสามเหลี่ยมเล็ก ...,/kaggle/input/image-processing-thai-language-i...
85238,ปลาดุกย่างเสียบไม้หลายตัวตั้งอยู่ในถาดมีผักแช่...,/kaggle/input/image-processing-thai-language-i...
85239,ปลาดุกปิ้งจำนวนมากที่วางอยู่ในถาดอะลูมิเนียมทร...,/kaggle/input/image-processing-thai-language-i...


# Train the model

In [16]:
instruction = "คิดแคปชั่นรูปนี้ให้หน่อย"
def convert_to_conversation(sample):
    conversation = [
        { "role": "user",
          "content" : [
            {"type" : "text",  "text"  : instruction},
            {"type" : "image", "image" : Image.open(sample["new_image_path"])} ]
        },
        { "role" : "assistant",
          "content" : [
            {"type" : "text",  "text"  : sample["caption_text"]} ]
        },
    ]
    return { "messages" : conversation }

In [20]:
converted_dataset = []
for index, row in tqdm(train_real_fr.iterrows()):
    converted_dataset.append(convert_to_conversation(row))
#converted_dataset = [convert_to_conversation(sample) for sample in train_real]

85241it [01:21, 1052.12it/s]


# real training

In [1]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [2]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    # "scb10x/typhoon2-qwen2vl-7b-vision-instruct",
    "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.1.8: Fast Qwen2_Vl vision patching. Transformers: 4.48.2.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/6.36G [00:00<?, ?B/s]

`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

In [21]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

Unsloth: Making `model.base_model.model.visual` require gradients


In [ ]:
model

In [23]:
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), # Must use!
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 150,
        # num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",     # For Weights and Biases

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        dataset_num_proc = 4,
        max_seq_length = 2048,
    ),
)

In [24]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 85,241 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 150
 "-____-"     Number of trainable parameters = 50,855,936
🦥 Unsloth needs about 1-3 minutes to load everything - please wait!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.597100
2,2.709900
3,2.634800
4,2.702500
5,2.613100
6,2.653400
7,2.264600
8,2.079400
9,1.845100
10,1.765400


# Process the test set

In [25]:
FastVisionModel.for_inference(model) # Enable for inference!

instruction = "คิดแคปชั่นรูปนี้ให้หน่อย"
test_cap = []
def generateCaption(image_path):
    image = Image.open(image_path)
    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": instruction}
        ]}
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens = False,
        return_tensors = "pt",
    ).to("cuda")
    
    output = model.generate(**inputs, max_new_tokens = 100,
                       use_cache = True, temperature = 0.5, min_p = 0.1)
    caption = tokenizer.decode(output[0], skip_special_tokens=True)
    test_cap.append(caption)

In [26]:
folder_path = '/kaggle/input/image-processing-thai-language-image-captioning/test/test'

image_paths = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
            image_paths.append(os.path.join(root, file))

print(len(image_paths))


2000


In [27]:
for img_path in tqdm(image_paths):
    generateCaption(img_path)

100%|██████████| 2000/2000 [3:10:31<00:00,  5.72s/it]  


In [28]:
test_cap[0]

'system\nYou are a helpful assistant.\nuser\nคิดแคปชั่นรูปนี้ให้หน่อย\nassistant\nแมลงที่มีหางยาวและมีลายสีขาวอยู่บนหัวและขา มีหางสีดำอยู่บนพื้น'

In [29]:
image_id = [k.split('/')[-1] for k in image_paths]

In [30]:
image_id_real = [k.split('.')[0] for k in image_id]

In [31]:
data = {'image_id': image_id_real, 'cc': test_cap}

# Create a DataFrame
submission = pd.DataFrame(data)

In [32]:
submission.to_csv("test_ans_1.csv")

In [33]:
submission['cc'] = submission['cc'].apply(lambda x: x.split('\n')[-1])


In [34]:
submission['cc'] = submission['cc'].apply(lambda x: x.replace('"',''))

In [35]:
submission

,image_id,cc
0,00767,แมลงที่มีหางยาวและมีลายสีขาวอยู่บนหัวและขา มีห...
1,00266,ป้ายสีดำตัวอักษรสีทองติดตั้งอยู่หน้าวัดสีขาวด้...
2,01496,เห็ดสีขาวที่อยู่บนพื้นดินที่มีใบไม้สีน้ำตาลและ...
3,01600,งูพันอยู่บนพื้นดินสีน้ำตาล มีใบไม้สีขาววางอยู่...
4,00847,ดอกไม้สีเหลืองที่มีใบสีเขียวอยู่ในสนามหญ้าและม...
...,...,...
1995,00319,แม่น้ำขนาดเล็กที่มีต้นไม้และพืชติดอยู่กับผนังแ...
1996,01863,ถ้วยไม้สีน้ำตาลที่ตั้งอยู่บนพื้นหญ้าสีเขียว
1997,01325,เสาสีขาวลายสีน้ำเงินตั้งอยู่บนบันไดสีน้ำตาล ด้...
1998,00222,ต้นไม้สีเขียวใบยาวตั้งอยู่ในถ้วยพลาสติกสีดำวาง...


In [36]:
submission.to_csv("first_sub1.csv")

# align labels

In [37]:
sample = pd.read_csv('/kaggle/input/image-processing-thai-language-image-captioning/sample_submission.csv', dtype={'image_id': str})

In [38]:
sampleid = sample['image_id']


In [39]:
id_list = sampleid.to_list()

In [40]:
id_list[:5]

['01354', '01413', '01802', '01243', '00693']

In [41]:
submission.head()

,image_id,cc
0,00767,แมลงที่มีหางยาวและมีลายสีขาวอยู่บนหัวและขา มีห...
1,00266,ป้ายสีดำตัวอักษรสีทองติดตั้งอยู่หน้าวัดสีขาวด้...
2,01496,เห็ดสีขาวที่อยู่บนพื้นดินที่มีใบไม้สีน้ำตาลและ...
3,01600,งูพันอยู่บนพื้นดินสีน้ำตาล มีใบไม้สีขาววางอยู่...
4,00847,ดอกไม้สีเหลืองที่มีใบสีเขียวอยู่ในสนามหญ้าและม...


In [42]:
df_final = submission.set_index('image_id').loc[id_list].reset_index()

In [43]:
df_final.head(30)

,image_id,cc
0,01354,ต้นไม้ที่มีใบสีขาว มีใบตั้งอยู่บนพื้นดินสีน้ำตาล
1,01413,กบตัวหนึ่งที่มีสีน้ำตาลและมีขนสีน้ำตาลตั้งอยู่...
2,01802,บ้านที่อยู่บนภูเขาขนาดใหญ่ที่มีต้นไม้และต้นไม้...
3,01243,สัตว์เลี้ยงลูกด้วยนมชนิดหนึ่งที่กำลังนั่งอยู่บ...
4,00693,แมลงที่มีลายสีน้ำตาลและลายสีขาวอยู่บนพื้นที่มี...
5,01695,ผัดหมูสับผักสีแดงสีเหลืองสีขาววางอยู่ในจานสีดำ
6,01735,กระถางสีน้ำตาลที่ทำจากเปลือกไม้ที่วางอยู่บนพื้...
7,01440,สัตว์ชนิดหนึ่งที่อยู่ในป่าและมีต้นไม้จำนวนมากอ...
8,00296,เห็ดที่มีสีขาวและมีสีน้ำตาล มีสีดำ มีสีน้ำเงิน...
9,01144,อาคารสีขาวมีหลังคาสีน้ำตาล มีป้ายสีน้ำตาลติดอย...


In [44]:
df_final = df_final.rename(columns={'cc': 'caption'})

In [45]:
df_final

,image_id,caption
0,01354,ต้นไม้ที่มีใบสีขาว มีใบตั้งอยู่บนพื้นดินสีน้ำตาล
1,01413,กบตัวหนึ่งที่มีสีน้ำตาลและมีขนสีน้ำตาลตั้งอยู่...
2,01802,บ้านที่อยู่บนภูเขาขนาดใหญ่ที่มีต้นไม้และต้นไม้...
3,01243,สัตว์เลี้ยงลูกด้วยนมชนิดหนึ่งที่กำลังนั่งอยู่บ...
4,00693,แมลงที่มีลายสีน้ำตาลและลายสีขาวอยู่บนพื้นที่มี...
...,...,...
1995,00029,รูปปั้นสัตว์ที่อยู่ภายในบ่อน้ำที่อยู่ภายในต้นไ...
1996,01065,ดอกไม้สีแดงที่มีใบสีเขียวอยู่บนต้นไม้สีน้ำตาล
1997,00195,ต้นไม้หลายต้นที่มีใบสีเขียวอยู่ในสนามหญ้าและมี...
1998,01026,ดอกไม้สีเหลืองติดอยู่กับต้นไม้ใบสีเขียวอยู่ในส...


In [46]:
df_final.to_csv('output.csv', index=False)

In [47]:
pd.read_csv('/kaggle/working/output.csv')

,image_id,caption
0,1354,ต้นไม้ที่มีใบสีขาว มีใบตั้งอยู่บนพื้นดินสีน้ำตาล
1,1413,กบตัวหนึ่งที่มีสีน้ำตาลและมีขนสีน้ำตาลตั้งอยู่...
2,1802,บ้านที่อยู่บนภูเขาขนาดใหญ่ที่มีต้นไม้และต้นไม้...
3,1243,สัตว์เลี้ยงลูกด้วยนมชนิดหนึ่งที่กำลังนั่งอยู่บ...
4,693,แมลงที่มีลายสีน้ำตาลและลายสีขาวอยู่บนพื้นที่มี...
...,...,...
1995,29,รูปปั้นสัตว์ที่อยู่ภายในบ่อน้ำที่อยู่ภายในต้นไ...
1996,1065,ดอกไม้สีแดงที่มีใบสีเขียวอยู่บนต้นไม้สีน้ำตาล
1997,195,ต้นไม้หลายต้นที่มีใบสีเขียวอยู่ในสนามหญ้าและมี...
1998,1026,ดอกไม้สีเหลืองติดอยู่กับต้นไม้ใบสีเขียวอยู่ในส...
